# 06 — Saving and Exporting

Once you have a CP you like, you'll want to send it somewhere — a plotter, a folder simulator, or a 3D printer. This notebook covers the export formats `eucare` ships with:

- `.heg` — eucare's native YAML serialization.
- SVG — vector for laser cutters / pen plotters.
- A high-level `overlap.save_results` that writes a whole result directory in one call (also shown in notebook 04).

STL via marching cubes lives behind the `[threed]` install extra; the FOLD format is tracked in `docs/improvements.md`.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, shrink_rotate_pattern


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = shrink_rotate_pattern(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)
print('SRG:', len(SRG.vertices), 'vertices,', len(SRG.faces), 'faces')
SRG.show(**rendering.CREASE_PATTERN_PRESET)


## .heg save

Eucare's native YAML serialization captures the full graph structure plus arbitrary attributes.

Note: the load path currently uses `yaml.SafeLoader`, so graphs with non-trivial Python attributes (tuples, numpy scalars) save but don't round-trip yet — see `docs/improvements.md` (P2.5).

In [ ]:
import tempfile, os
from eucare import io

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.heg')
    io.save_graph(path, SRG)
    size = os.path.getsize(path)
print(f'wrote {size} bytes')


## SVG export via `G.save(...)`

`G.save('out')` writes both `out.svg` and `out.png`. `G.show()` displays inline (vector SVG in Jupyter) without writing files. The SVG is the same vector drawing that `CairoRenderer` produced — open it in a browser or Inkscape for the full quality.

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    out = os.path.join(d, 'pattern')
    SRG.save(out, **rendering.CREASE_PATTERN_PRESET)
    print('files in temp dir:', sorted(os.listdir(d)))
    print('SVG head:')
    print(open(out + '.svg').read()[:200])

## Plotter-ready SVG via `SvgwriteRenderer`

For laser-cutter / pen-plotter pipelines the dedicated `SvgwriteRenderer` produces an SVG split into `{name}_borders.svg` / `{name}_interior.svg` (so you can use different tool heads for cut vs. score).

## All-in-one with `overlap.save_results`

If you've gone through `fold_complete`, `save_results(result, path)` writes the CP, both folded views, a back-lit composite, and a plotter-ready SVG in one call. See notebook 04 for an example.

## Other formats and tools

- **FOLD format** (`fold format.ipynb` in legacy notebooks) — the community standard for flat-foldable patterns. Not yet first-class in `eucare.io`; tracked in `docs/improvements.md`.
- **3D / STL** — `eucare.marching_cubes` + the optional `[threed]` install extra (`uv pip install -e '.[threed]'`).
- **Plotter hardware** — see `how_to_connect_plotter.txt` in the repo root for HP-GL setup tips.

## Wrap-up

That completes the core tour. The two trailing chapters add topical depth: [`07_Styling`](07_Styling.ipynb) for visual control and [`08_Modifications`](08_Modifications.ipynb) for ad-hoc graph surgery.